# Step 6 — Iterative Re-Search (Feedback Loop)

After Step 5 you have structured data — specific railroad company names, co-directors, and business partners. This step feeds those results back into the pipeline to fill gaps and deepen coverage.

**What this step does:**

1. **Compound term search** — for each person+railroad pair found in Step 5, generate targeted queries (e.g. `"John Smith" "Union Pacific"`) and re-run Internet Archive full-text search.
2. **Cross-reference search** — if two persons were both connected to the same railroad, search for their names together. Shared corporate ties often appear in the same documents.
3. **Poor's Manual search** — Poor's Manual of Railroads (1868+, annually) lists directors, officers, and financial data per company. Search IA for volumes covering identified railroads in the 1870s–1880s.

**Iteration plan:** run Steps 2–5, then this step, then re-run Steps 3a–5 with the newly discovered sources. At least two full passes are recommended before drawing conclusions.

In [ ]:
import sys, json, time
from pathlib import Path
from collections import defaultdict
import requests
from tqdm import tqdm

sys.path.insert(0, str(Path('.')))
from db import get_connection, set_progress, log_source, store_text, get_all_persons

conn = get_connection()
STEP = '6_feedback_loop'

## Analyze extraction results

In [ ]:
# Load all extractions
extractions = conn.execute("""
    SELECT e.research_id, p.canonical_name, p.known_states,
           e.railroad_company, e.connection_type, e.confidence, e.source_ref
    FROM extraction_results e
    JOIN persons p ON e.research_id = p.research_id
    WHERE e.railroad_company IS NOT NULL AND e.railroad_company != ''
      AND e.confidence != 'uncertain'
""").fetchall()

print(f"Confirmed railroad connections: {len(extractions)}\n")

# Railroad -> persons map
rr_to_persons = defaultdict(list)
person_to_rr  = defaultdict(list)
for r in extractions:
    rr_to_persons[r["railroad_company"]].append({
        "research_id": r["research_id"],
        "name": r["canonical_name"],
        "connection_type": r["connection_type"],
    })
    person_to_rr[r["research_id"]].append(r["railroad_company"])

# Find railroads with multiple persons in our dataset (highest cross-referencing value)
print("Railroads connecting 2+ persons in dataset:")
print(f"{'Railroad':<50} {'Persons':>7}")
print("-" * 60)
for rr, persons in sorted(rr_to_persons.items(), key=lambda x: -len(x[1])):
    if len(persons) >= 2:
        names = ", ".join(p["name"] for p in persons[:4])
        print(f"  {rr:<48} {len(persons):>5}  ({names})")

In [ ]:
# For each person + railroad pair, generate targeted search terms
search_terms = {}  # research_id -> list of new search terms

for research_id in set(e["research_id"] for e in extractions):
    person = conn.execute(
        "SELECT canonical_name FROM persons WHERE research_id=?", (research_id,)
    ).fetchone()
    name = person["canonical_name"]
    rrs = list(set(person_to_rr[research_id]))

    terms = []
    for rr in rrs[:5]:  # limit per person
        # Short railroad name: strip "Railway", "Railroad", "Company", etc.
        short_rr = rr
        for suffix in [' Railway', ' Railroad', ' Rail Road', ' Company', ' Co.', ' Co',
                       ' Corporation', ' & Co', ' & Company']:
            short_rr = short_rr.replace(suffix, '')
        short_rr = short_rr.strip()

        terms.append(f'"{name}" "{short_rr}"')
        terms.append(f'"{name}" railroad director')

    search_terms[research_id] = terms
    print(f"  {name}: {len(terms)} new search terms")

print(f"\nTotal: {sum(len(t) for t in search_terms.values())} new compound search terms")

## Re-run Internet Archive search with compound terms

In [ ]:
def search_ia_fulltext(query, rows=10):
    """Search Internet Archive full-text (same as step2_5)."""
    try:
        resp = requests.get(
            "https://archive.org/advancedsearch.php",
            params={"q": query, "fl[]": ["identifier", "title", "date", "description"],
                    "rows": rows, "output": "json"},
            headers={"User-Agent": "RailroadTiesPipeline/1.0"},
            timeout=30
        )
        resp.raise_for_status()
        return resp.json().get("response", {}).get("docs", [])
    except Exception as e:
        print(f"  IA search error: {e}")
        return []


# Run compound term searches on IA
new_sources_found = 0
for research_id, terms in tqdm(list(search_terms.items())[:50]):  # cap at 50 persons
    for term in terms[:4]:  # max 4 terms per person
        results = search_ia_fulltext(term, rows=5)
        for r in results:
            sid = log_source(
                conn, research_id, 'internet_archive',
                title=r.get("title", ""),
                url=f"https://archive.org/details/{r.get('identifier','')}",
                item_id=r.get("identifier", ""),
                snippet=str(r.get("description", ""))[:200],
                discovery_step='6_feedback_ia',
            )
            if sid:
                new_sources_found += 1
        time.sleep(0.5)

print(f"\nNew sources discovered via compound terms: {new_sources_found}")

## Cross-reference: persons sharing a railroad

In [ ]:
# Find pairs of persons sharing a railroad and search for them together
cross_searches_run = 0

for rr, persons in rr_to_persons.items():
    if len(persons) < 2:
        continue

    # Search for each pair in IA
    for i in range(len(persons)):
        for j in range(i+1, len(persons)):
            p1 = persons[i]
            p2 = persons[j]

            # Combined search
            query = f'"{p1["name"]}" "{p2["name"]}"'
            results = search_ia_fulltext(query, rows=5)

            for r in results:
                # Log for both persons
                for pid in [p1["research_id"], p2["research_id"]]:
                    log_source(conn, pid, 'internet_archive',
                               title=r.get("title", ""),
                               url=f"https://archive.org/details/{r.get('identifier','')}",
                               item_id=r.get("identifier", ""),
                               snippet=str(r.get("description", ""))[:200],
                               discovery_step='6_cross_reference',
                               relevance_note=f"co-found with {p1['name'] if pid==p2['research_id'] else p2['name']} via {rr}")

            cross_searches_run += 1
            time.sleep(0.5)

            if cross_searches_run >= 100:  # cap
                break
        if cross_searches_run >= 100:
            break

print(f"Cross-reference searches run: {cross_searches_run}")

## Check Poor's Manual for identified railroads

Poor's Manual of Railroads (annual 1868+) lists directors, officers, and financial data for each railroad company. Search Internet Archive for the specific years and companies identified in Step 5.

In [ ]:
# Get unique railroad companies from extractions
rr_companies = sorted(set(r["railroad_company"] for r in extractions if r["railroad_company"]))

print(f"Searching Poor's Manual for {len(rr_companies)} railroad companies\n")

poors_results = {}
for rr in rr_companies[:30]:  # limit
    # Short name for search
    short = rr.replace(' Railroad', '').replace(' Railway', '').replace(' Company', '').strip()
    query = f'poor manual railroad "{short}"'
    results = search_ia_fulltext(query, rows=5)

    if results:
        poors_results[rr] = results
        print(f"  {rr}: {len(results)} Poor's Manual hits")
        for r in results[:2]:
            print(f"    [{r.get('identifier','')}] {r.get('title','')} ({r.get('date','')})")

    time.sleep(0.8)

# Add Poor's Manual hits to sources_discovered for all relevant persons
for rr, results in poors_results.items():
    relevant_persons = [p["research_id"] for p in rr_to_persons.get(rr, [])]
    for result in results:
        for research_id in relevant_persons:
            log_source(conn, research_id, 'internet_archive',
                       title=result.get("title", ""),
                       url=f"https://archive.org/details/{result.get('identifier','')}",
                       item_id=result.get("identifier", ""),
                       discovery_step='6_poors_manual',
                       relevance_note=f"Poor's Manual match for railroad: {rr}")

print(f"\nPoor's Manual: found relevant volumes for {len(poors_results)} railroads")

## Mark progress and next steps

In [ ]:
# Mark all persons done for this step
persons = get_all_persons(conn)
for p in persons:
    rid = p["research_id"]
    new_sources = conn.execute(
        "SELECT COUNT(*) as n FROM sources_discovered WHERE research_id=? AND discovery_step LIKE '6%'",
        (rid,)
    ).fetchone()["n"]
    set_progress(conn, rid, STEP, 'done', result_count=new_sources)

print("=== Step 6 Summary ===\n")
print(f"Persons with railroad connections found in Step 5: {len(set(e['research_id'] for e in extractions))}")
print(f"Unique railroads identified: {len(rr_to_persons)}")
print(f"New sources discovered in Step 6: {conn.execute('SELECT COUNT(*) as n FROM sources_discovered WHERE discovery_step LIKE \"6%\"').fetchone()['n']}")
print()
print("NEXT: Download newly discovered IA volumes (re-run Step 3a) then re-run Step 5 extraction.")
print("The manifest should be updated with any new high-value volumes found in this step.")
print()
print("Volumes to add to manifest (discovered in Step 6):")
new_vols = conn.execute("""
    SELECT item_id, title, COUNT(DISTINCT research_id) as persons
    FROM sources_discovered WHERE discovery_step LIKE '6%' AND item_id IS NOT NULL
    GROUP BY item_id ORDER BY persons DESC LIMIT 20
""").fetchall()
for r in new_vols:
    print(f"  [{r['persons']:2d} persons] {r['item_id']:<40} {r['title'][:40]}")